# 2. Data validation

Run `python -m pip install -e ".[dev]"` from the repository root first. Each notebook runs independently and imports the same tested Python functions. Source timestamps retain their original local calendar meaning. Set `RETAIL_ETL_INPUT` to use another source and adjust `AS_OF` to its reporting cutoff.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from retail_etl.pipeline import project_root, read_source
from retail_etl.transform import prepare_data, feature_engineering, validate_output

ROOT = project_root()
SOURCE = Path(os.getenv("RETAIL_ETL_INPUT", str(ROOT / "data/raw/Online Retail.xlsx")))
OUTPUT = Path(os.getenv("RETAIL_ETL_OUTPUT", str(ROOT / "outputs")))
AS_OF = "2011-12-09"  # Inclusive end date of the bundled historical dataset.


## Apply the shared rules

Missing or extra columns fail the run. Row-level problems are quarantined with source row numbers and one or more reasons. Valid returns and anonymous sales are excluded from this particular analytical population.

In [ ]:
raw = read_source(SOURCE)
prepared = prepare_data(raw, as_of=AS_OF)
pd.Series(prepared.validation["rule_counts"]).sort_values(ascending=False)

## Reconcile every source row

Do not sum rule counts: a single record can fail several rules.

In [ ]:
counts = {k: prepared.validation[k] for k in ["raw_rows", "accepted_rows", "rejected_rows"]}
assert counts["raw_rows"] == counts["accepted_rows"] + counts["rejected_rows"]
counts

## Inspect excluded records

`source_row_number` is the original spreadsheet row number including its header. The raw source and excluded original values remain available.

In [ ]:
prepared.rejected.head(10)

## Verify final output

The publication gate checks the exact column order, required fields, source identities, and recalculated feature values. An entirely rejected input fails publication.

In [ ]:
final = feature_engineering(prepared.cleaned)
validate_output(final)